In [4]:
# Instalação das dependências necessárias no ambiente
# Nota: spacecore==0.4.2 é recomendado para total compatibilidade com sdplab==0.0.1
! pip install -q numpy matplotlib scipy "spacecore==0.4.2" sdplab cvxpy clarabel scs jax

# Problema das Marginais Quânticas (Quantum Marginal Problem - QMP) via SDP

O **Problema das Marginais Quânticas (Quantum Marginal Problem - QMP)** 

> **Definição:** Dado um conjunto de operadores de densidade reduzidos (marginais locais) $\{\omega_S\}_{S \in \mathcal{S}}$ definidos em subsistemas $S \subset \{1, \dots, N\}$, existe um estado quântico global $\rho$ de todo o sistema cuja redução a cada subsistema $S$ seja exatamente com $\omega_S$?
>
> $$\operatorname{Tr}_{S^c}(\rho) = \omega_S, \quad \forall S \in \mathcal{S}$$

-

### Estrutura deste Notebook

Neste notebook, abordaremos a resolução do QMP de forma modular e rigorosa:
1. **Estados Fundamentais e Projetores:** Representação de qubits e estados de Bell.
2. **Condições Físicas de Matrizes Densidade:** Hermiticidade, positividade semidefinida e traço unitário.
3. **Produto Tensorial:** Construção da família de estados GHZ.
4. **Traço Parcial:** Traço parcial via contração de índices (`np.einsum`).
5. **Base Ortonormal Hermitiana:** Base para o espaço $\mathrm{Herm}(d)$..
6. **Incorporação de Operadores Locais (`embed_operator`):** 
7. **Montagem do SDP via SDPLab (`build_qmp_sdp`):** Mapeamento do cone semidefinido.
8. **Resolução e Teste (`solve_qmp` e `verify_marginals`):** Execução do solver cônico (CLARABEL/SCS).
9. **Exemplos:**
   - **Exemplo 1 (GHZ):** Mixed = SIM, Pure = SIM.
   - **Exemplo 2 (Maximamente Misto):** Mixed = SIM, Pure = NÃO.
   - **Exemplo 3 (Incompatibilidade / Monogamia de Bell):** Mixed = NÃO, Pure = NÃO (com certificado dual de infactibilidade).

In [5]:
from __future__ import annotations

import sys
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

import numpy as np
import matplotlib.pyplot as plt

# Bibliotecas SpaceCore e SDPLab para Programação Semidefinida 
import spacecore
from spacecore import Context, DenseVectorSpace, HermitianSpace, NumpyOps
import sdplab
from sdplab import DenseConstraintOp, SDPProblem
from sdplab.solvers import run_cvxpy_solver

np.set_printoptions(precision=4, suppress=True)

print(f"Versão SDPLab:    {sdplab.__version__}")
print(f"Versão SpaceCore: {spacecore.__version__}")

Versão SDPLab:    0.0.1
Versão SpaceCore: 0.4.2


## 1. Definição de Estados e Operadores Elementares

Em computação quântica, o espaço de estados de um único qubit é o espaço de Hilbert bidimensional $\mathcal{H}_2 \cong \mathbb{C}^2$, com a base computacional ortonormal:

$$|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \quad |1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$

Dado um vetor de estado puro normalizado $|\psi\rangle$, o operador densidade associado é o projetor ortogonal de posto 1:

$$\rho = |\psi\rangle\langle\psi|$$

A função `ketbra(psi)` calcula este produto externo $\psi \psi^\dagger$.

In [ ]:
# Qubits da base computacional
zero = np.array([1.0, 0.0], dtype=complex)
one  = np.array([0.0, 1.0], dtype=complex)

def ketbra(psi: np.ndarray) -> np.ndarray:
    """
    Constrói o operador projetor |psi><psi| para um vetor de estado |psi>.
    """
    psi = np.asarray(psi, dtype=complex).reshape(-1)
    return np.outer(psi, psi.conj())


phi_plus = (np.kron(zero, zero) + np.kron(one, one)) / np.sqrt(2.0)
rho_phi_plus = ketbra(phi_plus)

print("Vetor de Bell |Phi+>:")
print(phi_plus)
print("\nMatriz densidade rho_Phi+ = |Phi+><Phi+| (4x4):")
print(rho_phi_plus)

Vetor de Bell |Phi+>:
[0.7071+0.j 0.    +0.j 0.    +0.j 0.7071+0.j]

Matriz densidade rho_Phi+ = |Phi+><Phi+| (4x4):
[[0.5+0.j 0. +0.j 0. +0.j 0.5+0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0.5+0.j 0. +0.j 0. +0.j 0.5+0.j]]


## 2. Condições de uma Matriz Densidade


1. **Hermiticidade:** $\rho = \rho^\dagger$. 
2. **Positividade Semidefinida:** $\rho \succeq 0 \iff \forall |\phi\rangle, \langle\phi|\rho|\phi\rangle \ge 0$ $\iff$  todos os autovalores de $\rho$ são reais e não negativos: $\lambda_i(\rho) \ge 0$.
3. **Normalização (Traço Unitário):** $\operatorname{Tr}(\rho) = \sum_i \lambda_i = 1$.

A função `verifica_densidade(rho, tol)` testa computacionalmente cada um desses requisitos.

In [ ]:
def verifica_densidade(rho: np.ndarray, tol: float = 1e-10) -> Dict[str, Any]: 
    rho = np.asarray(rho, dtype=complex)
    hermitiana = bool(np.allclose(rho, rho.conj().T, atol=tol))
    autovalores = np.linalg.eigvalsh((rho + rho.conj().T) / 2.0)
    positiva = bool(np.all(autovalores >= -tol))
    normalizada = bool(np.isclose(np.trace(rho), 1.0, atol=tol))
    
    return {
        "hermitiana": hermitiana,
        "positiva": positiva,
        "traco_1": normalizada,
        "autovalores": autovalores
    }

resultado_verif = verifica_densidade(rho_phi_plus)
print("Auditoria do estado rho_Phi+:")
for k, v in resultado_verif.items():
    print(f"  {k:12s}: {v}")

Auditoria do estado rho_Phi+:
  hermitiana  : True
  positiva    : True
  traco_1     : True
  autovalores : [0. 0. 0. 1.]


## 3. Produto Tensorial e o Estado GHZ



A função `tensor(*args)` realiza o produto de Kronecker múltiplos vetores:


### A Família de Estados GHZ (Greenberger-Horne-Zeilinger)
Para três qubits ($N=3$), o estado GHZ parametrizado por uma fase $\theta \in [0, 2\pi)$ é definido por:

$$|GHZ_\theta\rangle = \frac{1}{\sqrt{2}} (|000\rangle + e^{i\theta} |111\rangle)$$



In [ ]:
def tensor(*args: np.ndarray) -> np.ndarray:
    """
    Calcula o produto tensorial sequencial (Kronecker) de múltiplos operadores ou vetores.
    """
    resultado = args[0]
    for op in args[1:]:
        resultado = np.kron(resultado, op)
    return resultado

# Exemplo: Estado |000>
psi_000 = tensor(zero, zero, zero)
print(f"Dimensão do vetor |000>: {psi_000.shape}")
#Teste com o estado GHZ

def ghz(theta: float = 0.0) -> np.ndarray:
    estado = (
        tensor(zero, zero, zero)
        + np.exp(1j * theta) * tensor(one, one, one)
    ) / np.sqrt(2.0)
    return estado

psi_ghz = ghz(theta=0.0)
rho_ghz = ketbra(psi_ghz)

print(f"Matriz densidade rho_GHZ: formato = {rho_ghz.shape}")
print("Verificação de rho_GHZ:", verifica_densidade(rho_ghz))

Dimensão do vetor |000>: (8,)
Matriz densidade rho_GHZ: formato = (8, 8)
Verificação de rho_GHZ: {'hermitiana': True, 'positiva': True, 'traco_1': True, 'autovalores': array([0., 0., 0., 0., 0., 0., 0., 1.])}


## 4. Traço Parcial (`np.einsum`)

Dada uma matriz densidade global $\rho_{AB}$ de um sistema bipartido, o estado do subsistema $A$ isolado é obtido eliminando-se os graus de liberdade de $B$ por meio do **traço parcial**:

$$\rho_A = \operatorname{Tr}_B(\rho_{AB}) = \sum_{j} (I_A \otimes \langle j_B|) \rho_{AB} (I_A \otimes |j_B\rangle)$$

### Implementação Geral via Tensores
Para um sistema arbitrário de $N$ corpos com dimensões locais $d_0, d_1, \dots, d_{N-1}$, a matriz densidade global $\rho$ pode ser tratada como um tensor de ordem $2N$:

$$\rho_{a_0 a_1 \dots a_{N-1},\, b_0 b_1 \dots b_{N-1}}$$

onde os índices $a_k$ correspondem aos índices "bra" (linhas) e $b_k$ aos índices "ket" (colunas).

Para traçar fora um subsistema $c \in S^c$, impõe-se a contração $a_c = b_c$ e soma-se sobre este índice. Utilizando `np.einsum`, essa operação é realizada de forma vetorizada.

In [ ]:
def partial_trace(
    rho: np.ndarray,
    keep: Optional[Sequence[int]] = None,
    trace_out: Optional[Sequence[int]] = None,
    dims: Optional[Sequence[int]] = None,
) -> np.ndarray:
    rho = np.asarray(rho, dtype=complex)
    if dims is None:
        num_qubits = int(round(np.log2(rho.shape[0])))
        dims = [2] * num_qubits
    dims = tuple(dims)
    N = len(dims)

    if keep is not None and trace_out is not None:
        raise ValueError("Especifique apenas `keep` ou `trace_out`, não ambos.")
    if keep is not None:
        keep_systems = tuple(sorted(keep))
    elif trace_out is not None:
        trace_set = set(trace_out)
        keep_systems = tuple(sorted(i for i in range(N) if i not in trace_set))
    else:
        raise ValueError("É necessário especificar `keep` ou `trace_out`.")

    complement = tuple(i for i in range(N) if i not in keep_systems)

    # Tensor de ordem 2N: N índices bra e N índices ket
    rho_tensor = rho.reshape(dims + dims)

    bra_indices = list(range(N))
    ket_indices = list(range(N, 2 * N))

    # Para os subsistemas traçados fora, colapsa bra e ket no mesmo índice mudo
    for c in complement:
        ket_indices[c] = bra_indices[c]

    out_bra = [bra_indices[s] for s in keep_systems]
    out_ket = [ket_indices[s] for s in keep_systems]

    # Contração tensorial direta
    reduced_tensor = np.einsum(rho_tensor, bra_indices + ket_indices, out_bra + out_ket)
    d_out = int(np.prod([dims[s] for s in keep_systems]))
    return reduced_tensor.reshape(d_out, d_out)

# Calculando as marginais de dois qubits do estado |GHZ>
rho_AB_ghz = partial_trace(rho_ghz, keep=[0, 1], dims=[2, 2, 2])
rho_AC_ghz = partial_trace(rho_ghz, keep=[0, 2], dims=[2, 2, 2])
rho_BC_ghz = partial_trace(rho_ghz, keep=[1, 2], dims=[2, 2, 2])

print("Marginal rho_AB do estado GHZ (4x4):")
print(rho_AB_ghz)
print("\nAutovalores de rho_AB:", np.round(np.linalg.eigvalsh(rho_AB_ghz), 4))

Marginal rho_AB do estado GHZ (4x4):
[[0.5+0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0.5+0.j]]

Autovalores de rho_AB: [0.  0.  0.5 0.5]


## 5. Base Ortonormal no Espaço das Matrizes Hermitianas $\mathrm{Herm}(d)$

Para formular as restrições matriciais de traço parcial $\operatorname{Tr}_{S^c}(\rho) = \omega_S$ como igualdades escalares reais $\operatorname{Tr}(M_i \rho) = b_i$, precisamos projetar a igualdade matricial sobre uma base ortonormal do espaço vetorial real das matrizes Hermitianas $\mathrm{Herm}(d_S)$.

O espaço $\mathrm{Herm}(d)$ tem dimensão real $d^2$. Sob o **produto interno de Hilbert-Schmidt**:

$$\langle A, B \rangle_{\mathrm{HS}} = \operatorname{Tr}(A^\dagger B) = \operatorname{Tr}(A B) \quad (\text{para } A, B \in \mathrm{Herm}(d))$$

uma base ortonormal $\{E_k\}_{k=1}^{d^2}$ satisfaz $\operatorname{Tr}(E_k E_l) = \delta_{kl}$ e é construída por:
1. **$d$ elementos diagonais:**
   $$E_{ii} = |i\rangle\langle i|$$
2. **$\frac{d(d-1)}{2}$ elementos fora da diagonal simétricos (reais):**
   $$E_{ij}^{\mathrm{re}} = \frac{|i\rangle\langle j| + |j\rangle\langle i|}{\sqrt{2}}, \quad (i < j)$$
3. **$\frac{d(d-1)}{2}$ elementos fora da diagonal antissimétricos (imaginários):**
   $$E_{ij}^{\mathrm{im}} = \frac{i(|i\rangle\langle j| - |j\rangle\langle i|)}{\sqrt{2}}, \quad (i < j)$$

Total de elementos: $d + 2 \times \frac{d(d-1)}{2} = d^2$ matrizes Hermitianas ortonormais.

In [ ]:
def base_hermitiana(d: int) -> List[np.ndarray]:
    
    bases: List[np.ndarray] = []
    # Elementos diagonais
    for i in range(d):
        E_ii = np.zeros((d, d), dtype=complex)
        E_ii[i, i] = 1.0
        bases.append(E_ii)

    # Elementos fora da diagonal
    for i in range(d):
        for j in range(i + 1, d):
            # Parte real
            E_real = np.zeros((d, d), dtype=complex)
            E_real[i, j] = 1.0 / np.sqrt(2.0)
            E_real[j, i] = 1.0 / np.sqrt(2.0)
            bases.append(E_real)

            # Parte imaginária
            E_imag = np.zeros((d, d), dtype=complex)
            E_imag[i, j] = 1.0j / np.sqrt(2.0)
            E_imag[j, i] = -1.0j / np.sqrt(2.0)
            bases.append(E_imag)

    return bases

# Teste com d = 2 (um q-bit)
base_q = base_hermitiana(2)
print(f"Número de matrizes na base de Herm(2): {len(base_q)} (esperado: 2^2 = 4)")

# Teste de ortonormalidade: Matriz de Gram Tr(E_i E_j)
gram = np.array([[float(np.real(np.trace(e1 @ e2))) for e2 in base_q] for e1 in base_q])
print("Matriz de Gram Tr(E_k E_l) para d=2 (deve ser a identidade 4x4):")
print(gram)

Número de matrizes na base de Herm(2): 4 (esperado: 2^2 = 4)
Matriz de Gram Tr(E_k E_l) para d=2 (deve ser a identidade 4x4):
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


## 6. Extendendo Operadores Locais para o Espaço Total (`embed_operator`)

Como relacionar as restrições locais sobre $\omega_S$ diretamente com o estado global $\rho$?

Pela **dualidade do traço parcial** sob o produto interno de Hilbert-Schmidt:

$$\operatorname{Tr}[ E_k \operatorname{Tr}_{S^c}(\rho) ] = \operatorname{Tr}[ (E_k \otimes I_{S^c}) \rho ]$$

Portanto, a condição matricial de consistência marginal $\operatorname{Tr}_{S^c}(\rho) = \omega_S$ decompõe-se em $d_S^2$ restrições afins escalares sobre $\rho$:

$$\operatorname{Tr}(M_{S, k} \rho) = b_{S, k}, \quad \forall k=1, \dots, d_S^2$$

onde:
- $M_{S, k} = E_k \otimes I_{S^c}$ é o observável local $E_k$ **incorporado (embedded)** no espaço global $\mathcal{H}$;
- $b_{S, k} = \operatorname{Tr}(E_k \omega_S) \in \mathbb{R}$ é o valor esperado prescrito pela marginal.

A função `embed_operator(O_s, sistema, dim)` constrói este operador global $M_{S, k}$ posicionando as matrizes de identidade exatamente nos subsistemas complementares $S^c$.

In [11]:
def embed_operator(
    O_s: np.ndarray,
    sistema: Sequence[int],
    dim: Optional[Sequence[int]] = None,
    *,
    dims: Optional[Sequence[int]] = None,
) -> np.ndarray:
    r"""
    Incorpora um operador local O_s atuando no subsistema S (`sistema`) no espaço
    global H = prod_k H_k, inserindo identidades nos subsistemas do complemento S^c.
    """
    if dim is None:
        dim = dims
    if dim is None:
        raise ValueError("É necessário especificar as dimensões locais `dim` (ou `dims`).")
    dims_tuple = tuple(dim)
    Nt = len(dims_tuple)
    sistema_tuple = tuple(sistema)
    Ns = len(sistema_tuple)
    complemento = tuple(i for i in range(Nt) if i not in sistema_tuple)

    dimensao_s = tuple(dims_tuple[i] for i in sistema_tuple)
    O_tensor = O_s.reshape(dimensao_s + dimensao_s)

    # Produto externo com identidade para cada sistema complementar
    res = O_tensor
    for c in complemento:
        I_c = np.eye(dims_tuple[c], dtype=O_s.dtype)
        res = np.multiply.outer(res, I_c)

    # Mapeamento dos eixos originais para a ordem global
    axis_map_bra = {s: idx for idx, s in enumerate(sistema_tuple)}
    axis_map_ket = {s: Ns + idx for idx, s in enumerate(sistema_tuple)}
    for idx, c in enumerate(complemento):
        axis_map_bra[c] = 2 * Ns + 2 * idx
        axis_map_ket[c] = 2 * Ns + 2 * idx + 1

    perm = [axis_map_bra[i] for i in range(Nt)] + [axis_map_ket[i] for i in range(Nt)]
    res = np.transpose(res, perm)
    d_total = int(np.prod(dims_tuple))
    return res.reshape(d_total, d_total)

# Demonstração numérica da identidade de dualidade:
# Tr[(O_A (x) I_BC) rho_GHZ] == Tr[O_A Tr_BC(rho_GHZ)]
O_teste = np.array([[2.0, 1.0 - 1.0j], [1.0 + 1.0j, 3.0]], dtype=complex)
M_global_teste = embed_operator(O_teste, sistema=[0], dim=[2, 2, 2])

val_global = np.trace(M_global_teste @ rho_ghz)
val_local = np.trace(O_teste @ partial_trace(rho_ghz, keep=[0], dims=[2, 2, 2]))

print(f"Tr[(O_A (x) I_BC) rho_GHZ] = {val_global.real:.6f} + {val_global.imag:.1e}j")
print(f"Tr[O_A Tr_BC(rho_GHZ)]     = {val_local.real:.6f} + {val_local.imag:.1e}j")
print(f"Diferença absoluta:          {abs(val_global - val_local):.2e}")

Tr[(O_A (x) I_BC) rho_GHZ] = 2.500000 + 0.0e+00j
Tr[O_A Tr_BC(rho_GHZ)]     = 2.500000 + 0.0e+00j
Diferença absoluta:          0.00e+00


## 7. Construção do Problema SDP via SDPLab (`build_qmp_sdp`)

Com as restrições afins formuladas, podemos agora montar o problema no formato padrão cônico da biblioteca **SDPLab**:

$$\begin{aligned}
\min_{X \in \mathrm{Herm}(d)} \quad & \langle C, X \rangle = 0 \\
\text{sujeito a} \quad & \mathcal{A}(X) = b, \\
& X \succeq 0
\end{aligned}$$

onde:
- O espaço de domínio é $X = \rho \in \mathrm{Herm}(d_{\mathrm{total}})$;
- O vetor de custo $C = 0$ reflete que buscamos apenas a **factibilidade** do estado;
- O operador afim $\mathcal{A}$ empilha todas as restrições:
  1. **Normalização:** $\operatorname{Tr}(I_{d_{\mathrm{total}}} X) = 1.0$;
  2. **Consistência das Marginais:** $\operatorname{Tr}(M_{S, k} X) = b_{S, k}$.

### Detalhe do Pareamento de Frobenius no SDPLab
Na convenção interna da biblioteca `sdplab`, a ação de um operador de restrição densa $T$ sobre a matriz $X$ é dada pelo produto interno de Frobenius:

$$(\mathcal{A} X)_i = \sum_{p, q} T_{i, pq} X_{pq} = \operatorname{Tr}(T_i^T X)$$

Como desejamos medir $\operatorname{Tr}(M_i X) = b_i$ para matrizes Hermitianas $M_i$, fornecemos $T_i = M_i^T$ utilizando `np.swapaxes(M_full, -1, -2)`.

In [12]:
def parse_subsystem_key(key: Union[str, Sequence[int]], num_qubits: int = 3) -> Tuple[int, ...]:
    """
    Normaliza identificadores de subsistemas como 'AB', 'AC', 'BC' ou (0, 1), (0, 2).
    """
    if isinstance(key, (tuple, list)):
        return tuple(int(x) for x in key)
    if isinstance(key, str):
        char_map = {chr(ord('A') + i): i for i in range(26)}
        key_upper = key.upper().strip()
        return tuple(char_map[ch] for ch in key_upper if ch in char_map)
    raise TypeError(f"Formato de subsistema inválido: {key!r}")


def build_qmp_sdp(
    marginals_dict: Dict[Union[str, Tuple[int, ...]], np.ndarray],
    dims: Optional[Sequence[int]] = None,
    include_trace_norm: bool = True,
    ctx: Optional[Context] = None,
) -> Tuple[SDPProblem, np.ndarray, np.ndarray]:
    r"""
    Constrói a formulação de Programação Semidefinida (SDP) para o problema de
    representabilidade de marginais quânticas no formato nativo da biblioteca SDPLab.
    """
    # 1. Normalizar as chaves dos subsistemas e matrizes marginais
    normalized_marginals: Dict[Tuple[int, ...], np.ndarray] = {}
    for k, v in marginals_dict.items():
        sub_tuple = parse_subsystem_key(k)
        mat = np.asarray(v, dtype=complex)
        if mat.ndim != 2 or mat.shape[0] != mat.shape[1]:
            raise ValueError(f"A marginal {k} deve ser uma matriz quadrada; formato {mat.shape}.")
        normalized_marginals[sub_tuple] = mat

    if dims is None:
        max_idx = max(max(sub) for sub in normalized_marginals.keys())
        dims = [2] * (max_idx + 1)
    dims_tuple = tuple(dims)
    d_total = int(np.prod(dims_tuple))

    M_list: List[np.ndarray] = []
    b_list: List[float] = []

    # 2. Restrição de normalização: Tr(rho) = Tr(I_d rho) = 1.0
    if include_trace_norm:
        M_list.append(np.eye(d_total, dtype=complex))
        b_list.append(1.0)

    # 3. Restrições afins das marginais
    for sistema, omega_S in normalized_marginals.items():
        d_S = omega_S.shape[0]
        expected_d_S = int(np.prod([dims_tuple[s] for s in sistema]))
        if d_S != expected_d_S:
            raise ValueError(
                f"Dimensão da marginal para o subsistema {sistema} é {d_S}, "
                f"mas esperava-se {expected_d_S} conforme dims={dims_tuple}."
            )

        base_S = base_hermitiana(d_S)
        for E in base_S:
            val = float(np.real(np.trace(E @ omega_S)))
            M_global = embed_operator(E, sistema, dims_tuple)
            M_list.append(M_global)
            b_list.append(val)

    M_full = np.stack(M_list, axis=0)  # Formato (m, d_total, d_total)
    b_full = np.array(b_list, dtype=float)

    # 4. Configuração dos espaços no SDPLab
    if ctx is None:
        ctx = Context(NumpyOps(), dtype="complex128", check_level="none")

    dom = HermitianSpace(d_total, ctx=ctx)
    cod = DenseVectorSpace((len(b_full),), ctx=ctx)

    # Pareamento Frobenius: (A X)_i = Tr(T_i^T X) => T_i = M_i^T via swapaxes
    A = DenseConstraintOp(np.swapaxes(M_full, -1, -2), dom, cod, ctx)

    # Custo zero (C = 0) para o problema de factibilidade pura
    sdp = SDPProblem(dom.zeros(), A, b_full, ctx=ctx)

    return sdp, M_full, b_full

print("Funções parse_subsystem_key e build_qmp_sdp definidas com sucesso.")

Funções parse_subsystem_key e build_qmp_sdp definidas com sucesso.


## 8. Resolução (`solve_qmp`) e Teste Numérico (`verify_marginals`)

Para resolver o problema SDP formulado, utilizamos o backend `run_cvxpy_solver` do SDPLab, integrando com o solver **CLARABEL** ou opcionalmente **SCS**.

### Diagnóstico de Factibilidade e Infactibilidade
- **Problema Factível (`optimal`):** O solver encontra uma matriz densidade global $\rho \succeq 0$ que satisfaz todas as restrições com precisão numérica de máquina ($10^{-8} \sim 10^{-15}$).
- **Problema Infactível (`infeasible`):** Pelo **Teorema da Alternativa de Farkas** para cones semidefinidos, o solver cônico gera um **certificado dual de infactibilidade**, demonstrando rigorosamente a não existência de qualquer estado global compatível. Neste caso, uma exceção é tratada e capturada de forma elegante pela função `solve_qmp`.

### Teste Numérica (`verify_marginals` / `verifica_solucao`)
Ao recuperar uma matriz $\rho$, realizamos 6 testes de integridade física:
1. **Erro de Hermiticidade:** $\|\rho - \rho^\dagger\|_F$;
2. **Erro de Normalização:** $|\operatorname{Tr}(\rho) - 1|$;
3. **Condição de Positividade:** $\lambda_{\min}(\rho) \ge -\mathrm{tol}$;
4. **Espectro Completo:** Exibição de todos os autovalores $\lambda_i$;
5. **Pureza:** $\gamma = \operatorname{Tr}(\rho^2) \in [1/d, 1]$;
6. **Resíduo das Marginais:** Erro Frobenius $\|\operatorname{Tr}_{S^c}(\rho) - \omega_S\|_F$ e erro absoluto máximo para cada subsistema.

In [ ]:
def solve_qmp(
    sdp: SDPProblem,
    solver: str = "CLARABEL",
    verbose: bool = False,
    **kwargs: Any,
) -> Dict[str, Any]:
    
    try:
        X, y, prob = run_cvxpy_solver(
            sdp,
            solver=solver,
            verbose=verbose,
            return_problem=True,
            **kwargs,
        )
        status = prob.status
        is_feasible = (status in ("optimal", "optimal_inaccurate"))
        rho_mat = np.asarray(X)
        # Garante simetria hermitiana perfeita numérica
        rho_mat = (rho_mat + rho_mat.conj().T) / 2.0

        return {
            "status": status,
            "is_feasible": is_feasible,
            "rho": rho_mat,
            "dual_y": np.asarray(y),
            "problem": prob,
            "error_message": None,
        }
    except ValueError as err:
        err_str = str(err)
        status = "infeasible" if "infeasible" in err_str.lower() else "failed"
        return {
            "status": status,
            "is_feasible": False,
            "rho": None,
            "dual_y": None,
            "problem": None,
            "error_message": err_str,
        }


def verify_marginals(
    rho: Optional[np.ndarray],
    marginals_dict: Dict[Union[str, Tuple[int, ...]], np.ndarray],
    dims: Optional[Sequence[int]] = None,
    tol: float = 1e-7,
) -> Dict[str, Any]:
    
    if rho is None:
        return {"is_feasible": False, "message": "Nenhuma matriz rho fornecida (problema infactível)."}

    rho = np.asarray(rho, dtype=complex)
    if dims is None:
        num_qubits = int(round(np.log2(rho.shape[0])))
        dims = [2] * num_qubits
    dims = tuple(dims)

    herm_err = float(np.linalg.norm(rho - rho.conj().T, "fro"))
    tr_val = complex(np.trace(rho))
    tr_err = float(abs(tr_val - 1.0))

    rho_sym = (rho + rho.conj().T) / 2.0
    evals = np.linalg.eigvalsh(rho_sym)
    min_eval = float(np.min(evals))
    is_psd = bool(min_eval >= -tol)
    purity = float(np.real(np.trace(rho_sym @ rho_sym)))

    marginal_errors: Dict[str, Dict[str, Any]] = {}
    for k, omega in marginals_dict.items():
        sub_tuple = parse_subsystem_key(k)
        omega_np = np.asarray(omega, dtype=complex)
        calc_marginal = partial_trace(rho_sym, keep=sub_tuple, dims=dims)
        diff = calc_marginal - omega_np
        marginal_errors[str(k)] = {
            "frobenius_error": float(np.linalg.norm(diff, "fro")),
            "max_abs_error": float(np.max(np.abs(diff))),
            "calculated_marginal": calc_marginal,
            "target_marginal": omega_np,
        }

    return {
        "is_feasible": True,
        "hermitian_error": herm_err,
        "trace": tr_val,
        "trace_error": tr_err,
        "eigenvalues": evals,
        "min_eigenvalue": min_eval,
        "is_psd": is_psd,
        "purity": purity,
        "marginal_errors": marginal_errors,
    }




## 9. Exemplo 1: Marginais do Estado GHZ

### Descrição Física
Para o estado puro tripartite $|GHZ\rangle = \frac{|000\rangle + |111\rangle}{\sqrt{2}}$, o traço parcial sobre qualquer um dos qubits produz a mesma mistura clássica invariante de 2 qubits:

$$\omega_{AB} = \omega_{AC} = \omega_{BC} = \frac{1}{2}(|00\rangle\langle 00| + |11\rangle\langle 11|) = \begin{pmatrix} 0.5 & 0 & 0 & 0 \\ 0 & 0 & 0 & 0 \\ 0 & 0 & 0 & 0 \\ 0 & 0 & 0 & 0.5 \end{pmatrix}$$

### Regime Esperado
- **Mixed Representability:** **SIM** (factível). O SDP encontra uma matriz global válida $\rho_{ABC}$ (comumente a mistura clássica $\frac{1}{2}(|000\rangle\langle 000| + |111\rangle\langle 111|)$).
- **Pure Representability:** **SIM**, pois o estado puro $|GHZ\rangle$ existe e tem exatamente estas marginais.
- **Pureza esperada da mistura:** $\gamma = 0.5$, com dois autovalores iguais a $0.5$ e os demais nulos.

In [ ]:
# Montagem das marginais do estado GHZ
omega_ghz_2qubits = 0.5 * (ketbra(np.kron(zero, zero)) + ketbra(np.kron(one, one)))

marginals_ex1 = {
    "AB": omega_ghz_2qubits,
    "AC": omega_ghz_2qubits,
    "BC": omega_ghz_2qubits,
}

print("Exemplo 1 (GHZ)")
sdp_ex1, M_ex1, b_ex1 = build_qmp_sdp(marginals_ex1, dims=[2, 2, 2])
print(f"SDP construído: dimensão do domínio = {sdp_ex1.dom.n}x{sdp_ex1.dom.n}, restrições m = {len(b_ex1)}")

# Resolução com CLARABEL
res_ex1 = solve_qmp(sdp_ex1, solver="CLARABEL")
print(f"Status do Solver: {res_ex1['status']} | Factível: {res_ex1['is_feasible']}")

# Auditoria
verif_ex1 = verify_marginals(res_ex1["rho"], marginals_ex1, dims=[2, 2, 2])
print(f"Erro de Hermiticidade:  {verif_ex1['hermitian_error']:.2e}")
print(f"Traço Tr(rho):          {verif_ex1['trace'].real:.12f} (Erro: {verif_ex1['trace_error']:.2e})")
print(f"lambda_min:             {verif_ex1['min_eigenvalue']:.2e} -> rho >= 0: {verif_ex1['is_psd']}")
print(f"Pureza Tr(rho^2):       {verif_ex1['purity']:.4f}")
print("Autovalores de rho_ABC:")
print(" ", np.round(verif_ex1["eigenvalues"], 6))
print("Resíduos por marginal:")
for k, err in verif_ex1["marginal_errors"].items():
    print(f"  Marginal {k}: Erro Frobenius = {err['frobenius_error']:.2e}, Max Abs = {err['max_abs_error']:.2e}")

Iniciando montagem do SDP para o Exemplo 1 (GHZ)...
SDP construído: dimensão do domínio = 8x8, restrições m = 49
Status do Solver: optimal | Factível: True
Erro de Hermiticidade:  0.00e+00
Traço Tr(rho):          1.000000000000 (Erro: 2.44e-15)
lambda_min:             1.23e-15 -> rho >= 0: True
Pureza Tr(rho^2):       0.5000
Autovalores de rho_ABC:
  [0.  0.  0.  0.  0.  0.  0.5 0.5]
Resíduos por marginal:
  Marginal AB: Erro Frobenius = 4.57e-15, Max Abs = 2.82e-15
  Marginal AC: Erro Frobenius = 4.57e-15, Max Abs = 2.82e-15
  Marginal BC: Erro Frobenius = 4.57e-15, Max Abs = 2.82e-15


## 10. Exemplo 2: Marginais Maximamente Mistas

### Descrição Física
Prescrevemos que as três marginais bipartidas sejam maximamente mistas em 2 qubits:

$$\omega_{AB} = \omega_{AC} = \omega_{BC} = \frac{I_4}{4}$$

### Regime Esperado
- **Mixed Representability:** **SIM** (factível). O estado global maximamente misto $\rho_{ABC} = \frac{I_8}{8}$ satisfaz exatamente $\operatorname{Tr}_C(I_8 / 8) = I_4 / 4$.
- **Pure Representability:** **NÃO**.


In [ ]:
# Montagem das marginais maximamente mistas
omega_max_mixed = np.eye(4, dtype=complex) / 4.0

marginals_ex2 = {
    "AB": omega_max_mixed,
    "AC": omega_max_mixed,
    "BC": omega_max_mixed,
}

print("Exemplo 2 (Maximamente Misto)")
sdp_ex2, M_ex2, b_ex2 = build_qmp_sdp(marginals_ex2, dims=[2, 2, 2])
print(f"SDP construído: dimensão do domínio = {sdp_ex2.dom.n}x{sdp_ex2.dom.n}, restrições m = {len(b_ex2)}")

# Resolução com CLARABEL
res_ex2 = solve_qmp(sdp_ex2, solver="CLARABEL")
print(f"Status do Solver: {res_ex2['status']} | Factível: {res_ex2['is_feasible']}")

# Auditoria
verif_ex2 = verify_marginals(res_ex2["rho"], marginals_ex2, dims=[2, 2, 2])
print(f"Erro de Hermiticidade:  {verif_ex2['hermitian_error']:.2e}")
print(f"Traço Tr(rho):          {verif_ex2['trace'].real:.12f} (Erro: {verif_ex2['trace_error']:.2e})")
print(f"lambda_min:             {verif_ex2['min_eigenvalue']:.2e} -> rho >= 0: {verif_ex2['is_psd']}")
print(f"Pureza Tr(rho^2):       {verif_ex2['purity']:.4f} (Esperado para I_8/8: 0.1250)")
print("Autovalores de rho_ABC:")
print(" ", np.round(verif_ex2["eigenvalues"], 6))
print("Resíduos por marginal:")
for k, err in verif_ex2["marginal_errors"].items():
    print(f"  Marginal {k}: Erro Frobenius = {err['frobenius_error']:.2e}, Max Abs = {err['max_abs_error']:.2e}")

Iniciando montagem do SDP para o Exemplo 2 (Maximamente Misto)...
SDP construído: dimensão do domínio = 8x8, restrições m = 49
Status do Solver: optimal | Factível: True
Erro de Hermiticidade:  0.00e+00
Traço Tr(rho):          1.000000000000 (Erro: 0.00e+00)
lambda_min:             1.25e-01 -> rho >= 0: True
Pureza Tr(rho^2):       0.1250 (Esperado para I_8/8: 0.1250)
Autovalores de rho_ABC:
  [0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125]
Resíduos por marginal:
  Marginal AB: Erro Frobenius = 0.00e+00, Max Abs = 0.00e+00
  Marginal AC: Erro Frobenius = 0.00e+00, Max Abs = 0.00e+00
  Marginal BC: Erro Frobenius = 0.00e+00, Max Abs = 0.00e+00


## 11. Exemplo 3: Incompatibilidade de Marginais e Monogamia do Entrelaçamento

### Descrição Física
Tentamos impor que o qubit $A$ esteja **simultaneamente** maximamente entrelaçado em um estado puro de Bell $|\Phi^+\rangle$ com o qubit $B$ e com o qubit $C$:

$$\omega_{AB} = |\Phi^+\rangle\langle\Phi^+|, \quad \omega_{AC} = |\Phi^+\rangle\langle\Phi^+|$$

### Regime Esperado
- **Mixed Representability:** **NÃO** (infactível).
- **Pure Representability:** **NÃO** (infactível).


In [18]:
# Montagem das marginais incompatíveis de Bell
rho_bell = ketbra(phi_plus)

marginals_ex3 = {
    "AB": rho_bell,
    "AC": rho_bell,
}

#Vamos fazer um tete de consistência

rho_A_de_AB = partial_trace(marginals_ex3["AB"], keep=[0], dims=[2,2])
rho_A_de_AC = partial_trace(marginals_ex3["AC"], keep=[0], dims=[2,2])

overlap_error = np.linalg.norm(
    rho_A_de_AB - rho_A_de_AC,
    ord="fro"
)

print("\nTeste de consistência na sobreposição:")
print("Marginal de A obtida a partir de AB:")
print(rho_A_de_AB)

print("\nMarginal de A obtida a partir de AC:")
print(rho_A_de_AC)

print(f"\nErro de sobreposição ||rho_A^(AB) - rho_A^(AC)||_F = {overlap_error:.3e}")

if np.isclose(overlap_error, 0.0):
    print("A condição de sobreposição é satisfeita.")
    print("Entretanto, isso é apenas uma condição necessária, não suficiente,")
    print("para a existência de uma extensão global.")
else:
    print("A condição de sobreposição é violada.")


print("Exemplo 3 (Incompatibilidade de Bell)")
sdp_ex3, M_ex3, b_ex3 = build_qmp_sdp(marginals_ex3, dims=[2, 2, 2])
print(f"SDP construído: dimensão do domínio = {sdp_ex3.dom.n}x{sdp_ex3.dom.n}, restrições m = {len(b_ex3)}")

# Resolução com CLARABEL
res_ex3 = solve_qmp(sdp_ex3, solver="CLARABEL")
print(f"Status retornado pelo solver: {res_ex3['status']}")
print(f"O problema é factível?        {res_ex3['is_feasible']}")
print(f"Diagnóstico capturado:        {res_ex3['error_message']}")



Teste de consistência na sobreposição:
Marginal de A obtida a partir de AB:
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]

Marginal de A obtida a partir de AC:
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]

Erro de sobreposição ||rho_A^(AB) - rho_A^(AC)||_F = 0.000e+00
A condição de sobreposição é satisfeita.
Entretanto, isso é apenas uma condição necessária, não suficiente,
para a existência de uma extensão global.
Exemplo 3 (Incompatibilidade de Bell)
SDP construído: dimensão do domínio = 8x8, restrições m = 33
Status retornado pelo solver: infeasible
O problema é factível?        False
Diagnóstico capturado:        CLARABEL solver did not return a solution (infeasible).
